![Imgur](https://i.imgur.com/acSOZRh.png)

# Laboratorio n° 1. Parte C: Denoising Autoencoder

**Asignatura:** Redes Neuronales Profundas
**Bloque:** 1 -- Fundamentos de Deep Learning

---

## Introduccion

En este laboratorio vas a construir un **denoising autoencoder**: una red neuronal que aprende a eliminar ruido de imagenes.

Este ejercicio integra todos los conceptos vistos en los laboratorios anteriores: creacion de modelos con `nn.Sequential`, loops de entrenamiento, y evaluacion de redes.

---

## Instrucciones generales

- Las celdas de **setup** ya estan escritas -- ejecutalas sin modificar.
- Completa el codigo en las celdas marcadas con `# Tu codigo aqui`.
- Responde las preguntas de analisis en las celdas de texto.

---
## Denoising Autoencoder



---

### ¿Qué es un autoencoder?

Imaginá que tenés que fotocopiar un documento, pero la fotocopiadora agrega ruido (manchas, rayaduras). ¿Podría una red neuronal aprender a "limpiar" las fotocopias recibiendo como entrada la copia sucia y comparando la salida con el original?

Sí, y esa es exactamente la idea de un **denoising autoencoder** (autocodificador de eliminación de ruido).

Un autoencoder tiene dos partes:

```
imagen ruidosa → [ ENCODER ] → vector latente → [ DECODER ] → imagen reconstruida
```

**El Encoder** comprime la imagen en un vector de menor dimensión llamado **espacio latente**. La idea es que ese vector capture solo la información esencial de la imagen (la prenda, su forma, su textura), descartando el ruido, que es aleatorio y no tiene estructura.

**El Decoder** toma ese vector comprimido y reconstruye la imagen original sin ruido.

La red se entrena con la siguiente lógica:
- **Entrada:** imagen con ruido
- **Salida esperada (etiqueta):** imagen **sin** ruido
- **Función de pérdida:** error cuadrático medio (MSE) entre la imagen reconstruida y la original

Al minimizar ese error, la red aprende a comprimir solo la información relevante y a ignorar el ruido.

---

### Setup para la Parte 2

In [ ]:
# Setup de la Parte 2
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils import data
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
batch_size = 256

# Datos
mnist_train = torchvision.datasets.FashionMNIST(
    transform=transforms.ToTensor(), root="./data", train=True, download=True)
mnist_test = torchvision.datasets.FashionMNIST(
    transform=transforms.ToTensor(), root="./data", train=False, download=True)
iter_train = data.DataLoader(mnist_train, batch_size, shuffle=True, num_workers=2)
iter_valid = data.DataLoader(mnist_test, batch_size, shuffle=False, num_workers=2)

# Generador de ruido: Dropout apaga píxeles al azar con probabilidad p=0.5
# (cada píxel tiene un 50% de chance de ponerse en gris=0)
p = 0.5
noise = nn.Sequential(nn.Dropout(p))

# Visualizar el efecto del ruido
images, _ = next(iter(iter_train))
noise.train() # el Dropout solo aplica en modo train
noise_images = noise(images)

n = 8
fig, axes = plt.subplots(2, n, figsize=(n * 2, 4))
for i in range(n):
    axes[0, i].imshow(images[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(noise_images[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Con ruido", fontsize=12)
plt.suptitle("Imágenes originales vs. con ruido de Dropout")
plt.tight_layout()
plt.show()

### Ejercicio 1 — Construir el Encoder

**Objetivo:** Definir la mitad "compresora" del autoencoder.

El encoder debe:
1. Tomar como entrada una imagen de 28×28 píxeles (aplanada en un vector de 784 elementos).
2. Pasarla por una capa oculta de **100 neuronas** con activación ReLU.
3. Producir un vector latente de **30 elementos**.

Guardá el encoder en la variable `encoder`.

> **Pista:** Usá `nn.Sequential` con las capas en orden: `Flatten → Linear(784, 100) → ReLU → Linear(100, 30)`

In [ ]:
# Tu código aquí
encoder = None # reemplazá None con tu implementación

In [ ]:
# Test automático del encoder
images, _ = next(iter(iter_train))
try:
    latentes = encoder(images)
    assert latentes.shape[1] == 30, f"La salida del encoder debería tener 30 elementos, tiene {latentes.shape[1]}"
    print(f" Encoder OK: salida con forma {latentes.shape}")
except Exception as e:
    print(f" Error: {e}")

### Ejercicio 2 — Construir el Decoder

**Objetivo:** Definir la mitad "reconstructora" del autoencoder.

El decoder es el proceso inverso del encoder:
1. Recibe como entrada el vector latente de **30 elementos**.
2. Lo expande a través de una capa oculta de **100 neuronas** con activación ReLU.
3. Produce como salida una imagen reconstruida de **28×28** píxeles.

Guardá el decoder en la variable `decoder`.

> **Pista:** Usá `nn.Linear(30, 100) → ReLU → nn.Linear(100, 784) → Unflatten(1, (28, 28))`. La capa `nn.Unflatten(dim, shape)` convierte un vector en una matriz.

In [ ]:
# Tu código aquí
decoder = None # reemplazá None con tu implementación

In [ ]:
# Test automático del decoder
try:
    salidas = decoder(latentes)
    assert salidas.shape[1] == 28 and salidas.shape[2] == 28, \
        f"La salida del decoder debería ser (batch, 28, 28), es {salidas.shape}"
    print(f" Decoder OK: salida con forma {salidas.shape}")
except Exception as e:
    print(f" Error: {e}")

### Ejercicio 3 — Ensamblar el Autoencoder

**Objetivo:** Conectar el generador de ruido, el encoder y el decoder en una única red.

El autoencoder completo debe encadenar los tres bloques en orden:
```
imagen → [noise] → [encoder] → [decoder] → imagen reconstruida
```

Guardalo en la variable `net`.

> **Pista:** `nn.Sequential(noise, encoder, decoder)`

In [ ]:
# Tu código aquí
net = None # reemplazá None con tu implementación

**Pregunta de análisis:**

El bloque `noise` corrompe la imagen antes de pasarla al encoder. Sin embargo, la función de pérdida compara la salida del decoder con la imagen **original** (sin ruido). ¿Por qué esta asimetría es la clave para que la red aprenda a eliminar ruido?

*(Escribí tu respuesta acá)*

### Ejercicio 4 — Definir pérdida y optimizador

**Objetivo:** Elegir la función de pérdida adecuada para un autoencoder.

1. Instanciá la pérdida `nn.MSELoss()` y guardála en `loss_ae`.
2. Instanciá el optimizador `torch.optim.Adam(net.parameters())` y guardálo en `trainer_ae`.

In [ ]:
# Tu código aquí


**Pregunta de análisis:**

1. ¿Por qué usamos `MSELoss` y no `CrossEntropyLoss` para el autoencoder? ¿Cuál es la diferencia conceptual entre ambas?
2. Adam es un optimizador más sofisticado que SGD. ¿Qué ventaja tiene sobre SGD puro?

*(Escribí tu respuesta acá)*

### Ejercicio 5 — Entrenar el autoencoder

**Objetivo:** Implementar el loop de entrenamiento para el autoencoder.

Este bucle es similar al de la Parte 1, con una diferencia importante: **la etiqueta `y` no son clases, sino las imágenes originales sin ruido**. El objetivo es que la salida del modelo se parezca a la imagen de entrada.

1. Poné el modelo en modo entrenamiento (`net.train()`)
2. Para cada lote `(images, _)` del `iter_train`:
- Calculá la pérdida entre `net(images)` y las `images` originales
- Seguí los 4 pasos: `zero_grad → backward → step`
3. Entrenás por **50 épocas** (puede tardar unos minutos).
4. Visualizá las imágenes reconstruidas con la función `plot_reconstructions` provista.

In [ ]:
# Tu código aquí


In [ ]:
# Visualización de reconstrucciones
def plot_reconstructions(model, images, n=8):
    """
    Muestra imágenes originales, con ruido y reconstruidas por el modelo.

    Parámetros:
    model: el autoencoder entrenado
    images: lote de imágenes sin ruido
    n: número de imágenes a mostrar
    """
    ruido = nn.Sequential(nn.Dropout(0.5))
    ruido.train()
    noisy = ruido(images[:n])

    model.eval()
    with torch.no_grad():
        recon = model[1:](noisy) # pasar por encoder+decoder, sin el bloque de ruido
    model.train()

    fig, axes = plt.subplots(3, n, figsize=(n * 2, 6))
    for i in range(n):
        axes[0, i].imshow(images[i].squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(noisy[i].squeeze().detach(), cmap="gray")
        axes[1, i].axis("off")
        axes[2, i].imshow(recon[i].squeeze().detach(), cmap="gray")
        axes[2, i].axis("off")
    axes[0, 0].set_ylabel("Original", fontsize=11)
    axes[1, 0].set_ylabel("Con ruido", fontsize=11)
    axes[2, 0].set_ylabel("Reconstruida", fontsize=11)
    plt.suptitle("Autoencoder: eliminación de ruido")
    plt.tight_layout()
    plt.show()

images, _ = next(iter(iter_valid))
plot_reconstructions(net, images)

### Ejercicio 6 — Ruido gaussiano con un módulo personalizado

**Objetivo:** Reemplazar el bloque `noise` (basado en Dropout) por un módulo personalizado que agregue ruido gaussiano, usando herencia de `nn.Module`.

El ruido gaussiano agrega a cada píxel un valor aleatorio extraído de una distribución normal (en lugar de ponerlo directamente en cero como hace Dropout). Esto simula mejor el ruido de cámaras o sensores reales.

**Importante:** el ruido solo debe aplicarse durante el entrenamiento (cuando `self.training == True`). Durante la evaluación, las imágenes deben pasar sin modificación.

> **Pista:**
> - Creá una clase `GaussianNoise(nn.Module)` con parámetros `mean` y `std`.
> - En el método `forward`, verificá `self.training` para decidir si aplicar ruido.
> - El ruido gaussiano se genera con `torch.randn_like(tensor) * std + mean`.
> - Luego re-ensamblar el autoencoder con este nuevo bloque y re-entrenarlo.

In [ ]:
# Tu código aquí


**Pregunta de análisis:**

¿En qué se diferencia el ruido de Dropout del ruido gaussiano visualmente y conceptualmente? ¿Cuál te parece más realista para eliminar ruido en imágenes naturales?

*(Escribí tu respuesta acá)*

---
## Fin del Laboratorio 1c

Completaste el laboratorio sobre autoencoders. Ahora sabes como construir arquitecturas encoder-decoder para tareas de reconstruccion de señales.